In [9]:
# ! pip install ipykernel seaborn scikit-learn
# ! pip install matplotlib
# ! pip install seaborn
# ! pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score


import pandas as pd
import numpy as np
import os
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.base import clone

# 7 種指定模型
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

SIZE_GRID = {
    'mimic3c': [100, 500, 6000],
    # 'mimic3c': [100],
}
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [11]:
def create_csv(csv_path, dataframe):
    """
    Dataframe to CSV file 包含例外處理。
    """
    if os.path.exists(csv_path):
        try:
            os.remove(csv_path)
        except PermissionError as e:
            raise PermissionError(
                'csv file is currently being used by another program. Please close the file and try again.'
            ) from e
        
    dataframe.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'Exported: {csv_path}')
    print(f'  Shape: {dataframe.shape}')

In [12]:
def make_imbalanced_subset(X, y, n_total, imbalance_ratio=0.1, random_state=42):
    """
    從完整的二元資料中抽出大小為 n_total 的子集，強制使少數類佔 imbalance_ratio 比例。
    """
    n_minority = int(n_total * imbalance_ratio)
    n_majority = n_total - n_minority
    
    minority_idx = np.where(y == 1)[0]
    majority_idx = np.where(y == 0)[0]
    
    if len(minority_idx) < n_minority or len(majority_idx) < n_majority:
        raise ValueError("資料量不足以進行指定大小與比例的抽樣")
        
    np.random.seed(random_state)
    sampled_minority = np.random.choice(minority_idx, n_minority, replace=False)
    sampled_majority = np.random.choice(majority_idx, n_majority, replace=False)
    
    combined_idx = np.concatenate([sampled_minority, sampled_majority])
    np.random.shuffle(combined_idx) # 抽樣後打散順序
    
    if isinstance(X, pd.DataFrame):
        X_sub = X.iloc[combined_idx].copy()
        y_sub = y.iloc[combined_idx].copy()
    else:
        X_sub = X[combined_idx]
        y_sub = y[combined_idx]
        
    return X_sub, y_sub


In [13]:
def load_mimic3c(use_solution_a=True):
    """
    載入 MIMIC3C 資料集。
    若 use_solution_a=True，讀取 HW1 的 df_model_ready.csv（方案 A）。
    否則讀取 mimic3c.csv 自行前處理（方案 B）。
    """
    leakage_cols = [
        'ExpiredHospital',
        'LOSdays',
        'LOSgroupNum',
        'AdmitDiagnosis',
        'NumCallouts',
        'NumDiagnosis',
        'NumProcs',
        'NumCPTevents',
        'NumInput',
        'NumLabs',
        'NumMicroLabs',
        'NumNotes',
        'NumOutput',
        'NumRx',
        'NumProcEvents',
        'NumTransfers',
        'NumChartEvents',
        'TotalNumInteract',
        'hadm_id'
    ]
    
    if use_solution_a:
        df = pd.read_csv('data/df_model_ready.csv')
        # 防禦性移除標籤洩漏欄位
        df = df.drop(columns=[col for col in leakage_cols if col in df.columns])
        y = df['target']
        X = df.drop(columns=['target'])
    else:
        df = pd.read_csv('data/mimic3c.csv')
        id_cols = ['hadm_id']
        df = df.drop(columns=leakage_cols + id_cols, errors='ignore')
        
        # 簡易類別轉換與缺失值處理
        cat_cols = df.select_dtypes(include=['object']).columns
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
        df = df.fillna(df.median())
        
        y = df['ExpiredHospital']
        X = df.drop(columns=['ExpiredHospital'])
        
    print(f"X shape: {X.shape}, y=1 比例: {y.mean():.4f}, 最終納入特徵數量: {X.shape[1]}")
    return X, y

In [14]:
def fit_and_score(model_template, X_train, y_train, X_test, y_test):
    model = clone(model_template)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average='macro')

def run_one_dataset(X, y, n, n_repeats=20):
    seven_models = {
        'DecisionTree': DecisionTreeClassifier(max_depth=15, random_state=0),
        'KNN':          KNeighborsClassifier(n_neighbors=5),
        'LogisticReg':  LogisticRegression(max_iter=1000, random_state=0),
        'SVM':          SVC(kernel='rbf', gamma='scale', random_state=0),
        'MLP':          MLPClassifier(hidden_layer_sizes=(128, 128), max_iter=200, early_stopping=True, random_state=0),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=0),
        'NaiveBayes':   GaussianNB(),
    }
    
    rows = []
    for seed in range(n_repeats):
        for strategy in ['Stratified', 'Random']:
            stratify_param = y if strategy == 'Stratified' else None
            
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.2, random_state=seed, stratify=stratify_param
            )
            
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_te_scaled = scaler.transform(X_te)
            
            test_minority_ratio = np.mean(y_te == 1)
            
            for model_name, model_template in seven_models.items():
                f1 = fit_and_score(model_template, X_tr_scaled, y_tr, X_te_scaled, y_te)
                rows.append({
                    'seed': seed,
                    'strategy': strategy,
                    'model': model_name,
                    'n_samples': n,
                    'macro_f1': f1,
                    'test_minority_ratio': test_minority_ratio
                })
                
    return pd.DataFrame(rows)

In [15]:
def paired_significance_table(df_results):
    rows = []
    n_list = df_results['n_samples'].unique()
    models = df_results['model'].unique()
    
    for n in n_list:
        for model in models:
            sub_df = df_results[(df_results['n_samples'] == n) & (df_results['model'] == model)]
            
            f1_stratified = sub_df[sub_df['strategy'] == 'Stratified'].sort_values('seed')['macro_f1'].values
            f1_random = sub_df[sub_df['strategy'] == 'Random'].sort_values('seed')['macro_f1'].values
            
            t_val, p_val = stats.ttest_rel(f1_stratified, f1_random)
            mean_diff = np.mean(f1_stratified - f1_random)
            
            if mean_diff > 0 and p_val < 0.05:
                better = 'Stratified'
            elif mean_diff < 0 and p_val < 0.05:
                better = 'Random'
            else:
                better = 'n.s.'
                
            rows.append({
                'n': n, 'model': model, 't_value': t_val, 'p_value': p_val, 
                'mean_diff': mean_diff, 'better': better
            })
            
    return pd.DataFrame(rows)

def ceiling_effect_table(df_results, n_focus=500):
    sub_df = df_results[df_results['n_samples'] == n_focus]
    rows = []
    for model in sub_df['model'].unique():
        model_df = sub_df[sub_df['model'] == model]
        mean_f1 = model_df['macro_f1'].mean()
        
        f1_stratified = model_df[model_df['strategy'] == 'Stratified'].sort_values('seed')['macro_f1'].values
        f1_random = model_df[model_df['strategy'] == 'Random'].sort_values('seed')['macro_f1'].values
        diff_std = np.std(f1_stratified - f1_random, ddof=1)
        
        rows.append({'model': model, 'mimic3c_mean_f1': mean_f1, 'mimic3c_diff_std': diff_std})
        
    return pd.DataFrame(rows)

def std_ratio_table(df_results, n_focus=100):
    sub_df = df_results[df_results['n_samples'] == n_focus]
    rows = []
    for model in sub_df['model'].unique():
        model_df = sub_df[sub_df['model'] == model]
        f1_stratified = model_df[model_df['strategy'] == 'Stratified']['macro_f1'].values
        f1_random = model_df[model_df['strategy'] == 'Random']['macro_f1'].values
        
        ratio = np.std(f1_stratified, ddof=1) / np.std(f1_random, ddof=1)
        rows.append({'model': model, 'mimic3c_ratio': ratio})
        
    return pd.DataFrame(rows)

def plot_boxplot(df_results, n):
    sub_df = df_results[df_results['n_samples'] == n]
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=sub_df, x='model', y='macro_f1', hue='strategy', ax=ax)
    ax.set_title(f'Macro F1 by Model and Strategy (n={n})')

    fig.savefig(f'{OUTPUT_DIR}/mimic3c_boxplot_n{n}.png', bbox_inches='tight')
    plt.close(fig)

In [16]:
X_full, y_full = load_mimic3c(use_solution_a=True) # 請根據你的需求設定 True 或 False
all_dfs = []

for n in SIZE_GRID['mimic3c']:
    X_sub, y_sub = make_imbalanced_subset(X_full, y_full, n, imbalance_ratio=0.1, random_state=42)
    df = run_one_dataset(X_sub, y_sub, n, n_repeats=20)
    df['dataset'] = 'mimic3c'
    all_dfs.append(df)
    
df_results = pd.concat(all_dfs, ignore_index=True)

# 1. 輸出實驗總表
# df_results.('output_A2/mimic3c_results.csv', index=False)
create_csv(f'{OUTPUT_DIR}/mimic3c_results.csv', df_results)

# 2. 輸出 paired t-test 統計結果
df_sig_full = paired_significance_table(df_results)
# df_sig_full.('output_A2/significance_full.csv', index=False)
create_csv(f'{OUTPUT_DIR}/significance_full.csv', df_sig_full)

df_sig_filtered = df_sig_full[df_sig_full['p_value'] < 0.05]
with open(f'{OUTPUT_DIR}/significance_table.md', 'w') as f:
    f.write(df_sig_filtered.to_markdown(index=False))

# 3. 輸出 ceiling effect 檢查表
df_ceiling = ceiling_effect_table(df_results, n_focus=500)
with open(f'{OUTPUT_DIR}/ceiling_effect_table.md', 'w') as f:
    f.write(df_ceiling.to_markdown(index=False))

# 4. 輸出 std ratio 表
df_std_ratio = std_ratio_table(df_results, n_focus=100)
with open(f'{OUTPUT_DIR}/std_ratio_table.md', 'w') as f:
    f.write(df_std_ratio.to_markdown(index=False))

# 5. 輸出視覺化圖表
for n in SIZE_GRID['mimic3c']:
    plot_boxplot(df_results, n)

X shape: (58951, 25), y=1 比例: 0.0993, 最終納入特徵數量: 25
Exported: ./output/mimic3c_results.csv
  Shape: (280, 7)
Exported: ./output/significance_full.csv
  Shape: (7, 6)
